# NANDA clinical-reasoning SLR use case

This independent workflow retrieves scholarly literature on NANDA-I, NIC, NOC, clinical reasoning, and generative-AI agents. It does not use patient records, case notes, identifiers, or licensed terminology content.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'scripts' / '00_init_project.py').exists():
    raise RuntimeError('Run this notebook with the SLR-Engine repository as the working directory.')

project_id = 'NANDA'
project_root = repo_root / 'projects' / project_id
report_path = project_root / 'NandA-found-sources-on-clinical-reasoning-through-SLR.md'
print(f'Python: {sys.executable}')
print(f'Project: {project_root}')

Python: C:\Users\PROMET02\anaconda3\envs\prisma-env\python.exe
Project: D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\NANDA


## Initialize the independent project

The step is idempotent: it creates the project only when it does not already exist.

In [2]:
import subprocess

topic = ('Use of NANDA-I, NIC, and NOC terminologies to support clinical reasoning '
         'in research about generative-AI-based agents')
if not project_root.exists():
    subprocess.run([sys.executable, 'scripts/00_init_project.py', '--id', project_id, '--topic', topic], cwd=repo_root, check=True)
else:
    print('Project already exists; retaining its audit trail.')

Project already exists; retaining its audit trail.


In [3]:
import json
import yaml

config_path = project_root / 'project.yaml'
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
config.update({
    'topic': topic,
    'aim': 'Identify scholarly evidence on standardized nursing terminologies, clinical reasoning, and generative-AI agents.',
    'research_questions': ['What empirical evidence links NANDA-I, NIC, or NOC to clinical reasoning?', 'How are generative-AI or large-language-model agents evaluated alongside standardized nursing terminologies?', 'What reported safeguards, validation methods, and limitations affect clinical use?'],
    'question': topic,
    'languages': ['en'],
    'inclusion': [{'id': 'I1', 'text': 'Scholarly study, review, benchmark, or evaluation with a described method.'}, {'id': 'I2', 'text': 'Addresses NANDA-I, NIC, NOC, standardized nursing terminology, or clinical reasoning.'}, {'id': 'I3', 'text': 'Reports outcomes, validation, limitations, or implementation details.'}],
    'exclusion': [{'id': 'E1', 'text': 'Patient-specific advice, case notes, identifiers, or unpublished clinical data.'}, {'id': 'E2', 'text': 'Marketing or opinion-only material without a described scholarly method.'}, {'id': 'E3', 'text': 'Duplicate publication or insufficient bibliographic metadata.'}],
    'sources': {'openalex': True, 'crossref': True, 'pubmed': True, 'europe_pmc': True, 'arxiv': False, 'semantic_scholar': True, 'dblp': False, 'ia_scholar': False, 'core': False, 'scopus': 'manual', 'web_of_science': 'manual'},
    'llm': {'provider': 'agent'}
})
config_path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=False), encoding='utf-8')

queries = project_root / 'queries'
query_text = '(NANDA OR \"NANDA-I\" OR NIC OR NOC OR \"standardized nursing terminology\") AND (\"clinical reasoning\" OR \"clinical decision making\") AND (\"generative AI\" OR \"large language model\" OR LLM OR agent)'
(queries / 'openalex.txt').write_text(query_text, encoding='utf-8')
(queries / 'pubmed.txt').write_text(query_text, encoding='utf-8')
(queries / 'europepmc.txt').write_text(query_text, encoding='utf-8')
(queries / 'semantic_scholar.txt').write_text(query_text, encoding='utf-8')
(queries / 'crossref.json').write_text(json.dumps({'query.bibliographic': query_text, 'rows': 100}), encoding='utf-8')
print(config_path)
print(query_text)

D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\NANDA\project.yaml
(NANDA OR "NANDA-I" OR NIC OR NOC OR "standardized nursing terminology") AND ("clinical reasoning" OR "clinical decision making") AND ("generative AI" OR "large language model" OR LLM OR agent)


## Search and deduplicate

This runs only public scholarly metadata APIs. Results remain in the project database for human title/abstract screening.

In [4]:
search_command = [sys.executable, 'scripts/02_search_open.py', '--project', project_id, '--sources', 'openalex,crossref,pubmed,europe_pmc,semantic_scholar', '--max-records', '100', '--acknowledge-warnings']
search_result = subprocess.run(search_command, cwd=repo_root, text=True, capture_output=True)
print(search_result.stdout)
if search_result.returncode:
    print(search_result.stderr)
    raise RuntimeError(f'Search failed with exit code {search_result.returncode}.')

dedup_result = subprocess.run([sys.executable, 'scripts/03_dedup.py', '--project', project_id, '--acknowledge-warnings'], cwd=repo_root, text=True, capture_output=True)
print(dedup_result.stdout)
if dedup_result.returncode:
    print(dedup_result.stderr)
    raise RuntimeError(f'Deduplication failed with exit code {dedup_result.returncode}.')

[warn] OpenAlex: no OPENALEX_API_KEY set. Long Boolean queries
       may fail with HTTP 400. Set OPENALEX_API_KEY in .env or
       openalex_api_key in project.yaml.
Validating queries...

[openalex]
  ok

[crossref]
  WARN:  query.bibliographic is a long flat string. Crossref's relevance ranking on long strings can pull in tangentially-related literatures. Consider splitting into multiple narrower groups (list of strings) — Crossref AND's repeated query.bibliographic params for ranking.
  WARN:  no `filter` block. Filters (type, date range, has-abstract, container-title) are STRICT in Crossref — they actually exclude non-matching records. Consider at minimum a `filter.type` and the engine's date filters.

[pubmed]
  ok

[europe_pmc]
  ok

[semantic_scholar]
  ok

[run] openalex: query_id=openalex_20260816_063243
  openalex: 50 records...
  openalex: 100 records...
[done] openalex: 100 records
[run] crossref: query_id=crossref_20260816_063246
  crossref: 50 records...
  crossref: 100 

Records before: 289
Records after:  283
Fuzzy merges:   6 (examined 34 pairs)
Source hits:    294

Next: python scripts/04_screen_prep.py --project NANDA



## Export the minimal output-derived evidence report

Ranking is a transparent lexical triage aid, not a screening decision. WILLMA may be used later only to advise on one scholarly candidate at a time; it cannot write decisions to this project.

In [ ]:
import sqlite3
import json
import re
from urllib.parse import quote

import requests

with sqlite3.connect(project_root / 'project.db') as connection:
    connection.row_factory = sqlite3.Row
    total_records = connection.execute('SELECT COUNT(*) FROM records').fetchone()[0]
    records = connection.execute('SELECT records.title, records.year, records.doi, records.abstract FROM records WHERE records.doi IS NOT NULL AND TRIM(records.doi) <> \"\"').fetchall()

keywords = ('nanda', 'nic', 'noc', 'standardized nursing terminology', 'clinical reasoning', 'clinical decision', 'generative ai', 'large language model', 'llm', 'agent')
def matches_term(haystack, term):
    pattern = r'(?<!\w)' + re.escape(term) + r'(?:s)?(?!\w)' if term in ('llm', 'agent') else r'(?<!\w)' + re.escape(term) + r'(?!\w)'
    return bool(re.search(pattern, haystack))

def relevance_score(record):
    haystack = ' '.join(str(record[field] or '') for field in ('title', 'abstract')).lower()
    return sum(matches_term(haystack, keyword) for keyword in keywords)

domain_terms = {'NANDA/NIC/NOC terminology': ('nanda', 'nanda-i', 'nic', 'noc', 'standardized nursing terminology'), 'Clinical reasoning': ('clinical reasoning', 'clinical decision'), 'AI or agents': ('generative ai', 'large language model', 'llm', 'agent')}
def domain_coverage(record):
    haystack = ' '.join(str(record[field] or '') for field in ('title', 'abstract')).lower()
    return [label for label, terms in domain_terms.items() if any(matches_term(haystack, term) for term in terms)]

ranked_records = sorted(records, key=lambda record: (-relevance_score(record), -(record['year'] or 0)))
def initials(given):
    return ' '.join(f'{part[0]}.' for part in given.replace('-', ' ').split() if part)

def apa_authors(authors):
    names = []
    for author in authors[:20]:
        family = author.get('family') or author.get('name')
        if family:
            given = initials(author.get('given', ''))
            names.append(f'{family}, {given}'.rstrip(', '))
    if not names:
        return None
    if len(names) == 1:
        return names[0]
    return ', '.join(names[:-1]) + ', & ' + names[-1]

def crossref_work(doi):
    encoded_doi = quote(doi, safe='')
    response = requests.get('https://api.crossref.org/works/' + encoded_doi, timeout=(10, 30))
    response.raise_for_status()
    return response.json()['message']

def apa_reference(work, doi):
    authors = apa_authors(work.get('author', []))
    title = ' '.join(work.get('title', [])).strip()
    issued = work.get('issued', {}).get('date-parts', [[None]])[0][0]
    journal = ' '.join(work.get('container-title', [])).strip()
    if not all((authors, title, issued, journal)):
        return None
    volume = work.get('volume', '')
    issue = work.get('issue', '')
    pages = work.get('page', '')
    journal_part = f'*{journal}*'
    if volume:
        journal_part += f', *{volume}*'
    if issue:
        journal_part += f'({issue})'
    if pages:
        journal_part += f', {pages}'
    doi_url = f'https://doi.org/{doi}'
    return f'{authors} ({issued}). {title}. {journal_part}. [{doi_url}]({doi_url})'

verified_records = []
for record in ranked_records:
    try:
        work = crossref_work(record['doi'])
        reference = apa_reference(work, record['doi'])
    except requests.RequestException:
        reference = None
    if reference:
        verified_records.append((record, reference))
    if len(verified_records) == 30:
        break

query_files = [('OpenAlex', 'openalex.txt'), ('Crossref', 'crossref.json'), ('PubMed', 'pubmed.txt'), ('Europe PMC', 'europepmc.txt'), ('Semantic Scholar', 'semantic_scholar.txt')]
query_rows = []
for source, filename in query_files:
    content = (project_root / 'queries' / filename).read_text(encoding='utf-8').strip()
    if filename.endswith('.json'):
        payload = json.loads(content)
        query = payload['query.bibliographic']
        if payload.get('rows'):
            query += f" (rows: {payload['rows']})"
    else:
        query = content
    query_rows.append(f'| {source} | `{query.replace(chr(124), chr(92) + chr(124))}` |')

score_counts = {score: sum(relevance_score(record) == score for record, _ in verified_records) for score in sorted({relevance_score(record) for record, _ in verified_records}, reverse=True)}
domain_counts = {label: sum(label in domain_coverage(record) for record, _ in verified_records) for label in domain_terms}
score_summary_rows = []
for score, count in score_counts.items():
    interpretation = 'Broadest protocol-term overlap' if score >= 5 else 'Strong multi-term overlap' if score >= 4 else 'Focused overlap; inspect the reference before screening'
    score_summary_rows.append(f'| {score} | {count} | {interpretation} |')

lines = ['<style>', 'body, body * { font-size: 12px !important; }', '</style>', '', '# NandA Found Sources on Clinical Reasoning Through SLR', '', '- [Search Method](#search-method)', '- [Search Queries](#search-queries)', '- [Relevance Ranking](#relevance-ranking)', '- [Relevant Sources Found Through SLR](#relevant-sources-found-through-slr)', '- [Appendix: How This Report Was Produced](#appendix-how-this-report-was-produced)', '', '## Scope', 'Scholarly metadata retrieved for the independent NANDA project on NANDA-I, NIC, NOC, clinical reasoning, and generative-AI-based agents. This report contains candidate sources only; inclusion requires human screening and it contains no patient-specific data.', '', '## Search Method', 'The workflow queried OpenAlex, Crossref, PubMed, Europe PMC, and Semantic Scholar using the literal query retained in `projects/NANDA/queries/`. It stored returned metadata in `project.db` and performed the engine deduplication stage before this export.', '', '## Search Queries', '| Source | Query |', '| --- | --- |', *query_rows, '', '## Relevance Ranking', 'Candidates are ordered by the number of protocol terms found in their title and abstract, followed by publication year. This deterministic ranking supports triage only and is not an inclusion, exclusion, or clinical decision.', '', '| Rank | Score | Candidate | Verified DOI |', '| ---: | ---: | --- | --- |']
lines = lines[:-2]
lines.extend(['### Human-Readable Triage Overview', f'The {len(verified_records)} displayed references are Crossref-verified candidates. The score counts how many of 10 protocol terms occur in the title or abstract; a higher score indicates broader wording overlap with the query, not stronger evidence or eligibility.', '', '| Score | Displayed references | Interpretation |', '| ---: | ---: | --- |', *score_summary_rows, '', '| Protocol domain | Displayed references mentioning the domain |', '| --- | ---: |', *[f'| {label} | {count} |' for label, count in domain_counts.items()], '', 'Use the **Reference** link in the table to jump to the complete APA-style citation below. Domain labels show which of the three protocol concepts are mentioned in the candidate metadata.', '', '| Rank | Reference | Score | Protocol domains found | Candidate |', '| ---: | ---: | ---: | --- | --- |'])
for index, (record, _) in enumerate(verified_records, start=1):
    candidate_title = ' '.join(str(record["title"] or 'Untitled').split()).replace('|', '\\|')
    coverage = '; '.join(domain_coverage(record)) or 'No protocol domain detected'
    lines.append(f'| {index} | [{index}](#source-{index}) | {relevance_score(record)} | {coverage} | {candidate_title} |')
lines.extend(['', '## Relevant Sources Found Through SLR', 'All references below were verified against the Crossref work record at export time.'])
for index, (_, reference) in enumerate(verified_records, start=1):
    lines.extend(['', f'<a id="source-{index}"></a>', f'{index}. {reference}'])
if not verified_records:
    lines.append('No candidate records with Crossref-verified APA metadata were available during this run.')
appendix_lines = ['', '## Appendix: How This Report Was Produced', '', 'This appendix documents the reproducible path from the NANDA project configuration to this Markdown report. It describes candidate discovery and bibliographic presentation; it does not represent completed human screening or clinical validation.', '', '### Inputs and Purpose', '', '| Component | What was used | Why it was used |', '| --- | --- | --- |', '| Study protocol | `projects/NANDA/project.yaml` | Defines the topic, research questions, source enablement, and inclusion/exclusion criteria. |', '| Search definitions | Five files in `projects/NANDA/queries/` | Preserves the literal source-specific search strings shown above. |', f'| Candidate database | `projects/NANDA/project.db` with {total_records} deduplicated records at export | Supplies the scholarly metadata considered by the report. |', '| Export implementation | `SLR_Engine_SURF_AI_HUB_NANDA-Use-Case.ipynb` | Produces the ranking table, Crossref verification, APA-style references, and this report. |', '| Crossref work API | `https://api.crossref.org/works/{DOI}` | Verifies the metadata required to render a displayed reference. |', '', '### Workflow', '', '```mermaid', 'flowchart TD', '    A[Define NANDA project protocol] --> B[Create source-specific query files]', '    B --> C[Search OpenAlex, Crossref, PubMed, Europe PMC, and Semantic Scholar]', '    C --> D[Store candidate metadata in project.db]', '    D --> E[Run SLR-Engine deduplication]', '    E --> F[Export the Markdown report]', '    F --> G[Human screening and evidence appraisal]', '```', '', 'The first six stages were used to create this report. The final stage remains a required human activity and is outside the scope of this export.', '', '### Ranking and Reference Assembly', '', '```mermaid', 'flowchart TD', '    A[Deduplicated candidate records] --> B[Keep records with a non-empty DOI]', '    B --> C[Count whole protocol terms and phrases in title and abstract]', '    C --> D[Sort by score, then publication year]', '    D --> E[Request Crossref work metadata for each DOI]', '    E --> F{Author, title, year, and container title available?}', '    F -- Yes --> G[Render APA-style reference with DOI link]', '    F -- No or request fails --> H[Skip from displayed reference list]', '    G --> I[Stop after 30 verified references]', '```', '', '### Processing Rules', '', '| Step | Rule applied | Rationale |', '| --- | --- | --- |', '| Candidate selection | Only records with a non-empty DOI enter the displayed reference pipeline. | A DOI permits a direct Crossref work lookup and a clickable resolver link. |', '| Relevance score | Counts whole protocol terms and phrases in title plus abstract; ties are ordered by newer publication year. | Prevents short terms such as `NIC` from matching inside unrelated words, while retaining deterministic triage. |', '| Crossref verification | A reference is shown only when Crossref returns author, title, year, and container title. | Avoids rendering incomplete local bibliographic metadata as a final reference. |', '| Display limit | Export stops after 30 verified references. | Keeps the report readable while retaining a reproducible selection rule. |', '', '### Interpretation Boundaries', '', '| This report does | This report does not |', '| --- | --- |', '| Preserve the configured queries, candidate ranking, DOI links, and Crossref-verified citation metadata. | Conduct title/abstract screening, full-text appraisal, risk-of-bias assessment, or data extraction. |', '| Support transparent triage of scholarly candidate records. | Treat the lexical score as an inclusion/exclusion decision, evidence-quality measure, or clinical recommendation. |', '| Use scholarly metadata and public Crossref bibliographic responses. | Process patient records, identifiers, clinical notes, unpublished data, or licensed terminology content. |', '', 'To reproduce the report, run the NANDA notebook in the configured Python environment. The export reads the current project database and query files, so changes to either input will be reflected in a subsequent report.']
workflow_start = appendix_lines.index('### Workflow')
processing_start = appendix_lines.index('### Processing Rules')
data_science_appendix = ['### Executable Components', '', '| File | Executed responsibility | Output or effect |', '| --- | --- | --- |', '| `SLR_Engine_SURF_AI_HUB_NANDA-Use-Case.ipynb` | Orchestrates project setup, configuration, search, deduplication, feature scoring, Crossref checks, and report writing. | This Markdown report. |', '| `scripts/00_init_project.py` | Creates `projects/NANDA/` when it does not already exist. | Initial project structure and audit trail. |', '| `scripts/02_search_open.py` | Queries OpenAlex, Crossref, PubMed, Europe PMC, and Semantic Scholar. | Candidate metadata written to the project database. |', '| `scripts/03_dedup.py` | Applies the SLR-Engine deduplication stage. | Deduplicated candidate records in `project.db`. |', '', '### Data Lineage: Code to Report', '', '```mermaid', 'flowchart LR', '    NB[SLR_Engine_SURF_AI_HUB_NANDA-Use-Case.ipynb]', '    INIT[scripts/00_init_project.py]', '    SEARCH[scripts/02_search_open.py]', '    DEDUP[scripts/03_dedup.py]', '    subgraph PROJECT[projects/NANDA]', '        CONFIG[project.yaml]', '        QUERIES[queries/*.txt and crossref.json]', f'        DB[project.db\n{total_records} deduplicated records]', '        REPORT[NandA-found-sources-on-clinical-reasoning-through-SLR.md]', '    end', '    NB --> INIT --> PROJECT', '    NB --> CONFIG', '    NB --> QUERIES', '    NB --> SEARCH', '    QUERIES --> SEARCH', '    SEARCH --> DB', '    NB --> DEDUP', '    DB --> DEDUP --> DB', '    NB --> REPORT', '    DB --> REPORT', '```', '', '### Data Science Path: Metadata to Human Triage', '', '```mermaid', 'flowchart TD', f'    A[project.db\n{total_records} deduplicated records] --> B[Keep records with DOI\n{len(records)} DOI-bearing candidates]', '    B --> C[Text feature engineering\ncombine title + abstract; lowercase]', '    C --> D[Boundary-aware term matching\n10 protocol terms and phrases]', '    D --> E[Feature outputs\nlexical score + three domain flags]', '    E --> F[Rank by score descending\nthen publication year descending]', '    F --> G[Crossref work lookup per DOI]', '    G --> H{Author, title, year, and container title present?}', '    H -- Yes --> I[APA-style citation + DOI link]', '    H -- No or request failure --> J[Not displayed as a verified reference]', f'    I --> K[Displayed list\n{len(verified_records)} Crossref-verified references]', '    K --> L[Human screening, appraisal, and synthesis]', '```', '', 'The data-science transformation creates transparent metadata features only. It does not infer study quality, causal validity, or clinical suitability.', '', '### Project Map', '', '```mermaid', 'flowchart TB', '    ROOT[SLR-Engine/]', '    ROOT --> NOTEBOOK[SLR_Engine_SURF_AI_HUB_NANDA-Use-Case.ipynb]', '    ROOT --> SCRIPTS[scripts/]', '    SCRIPTS --> INIT_FILE[00_init_project.py]', '    SCRIPTS --> SEARCH_FILE[02_search_open.py]', '    SCRIPTS --> DEDUP_FILE[03_dedup.py]', '    ROOT --> PROJECT_ROOT[projects/NANDA/]', '    PROJECT_ROOT --> YAML[project.yaml]', '    PROJECT_ROOT --> QUERY_DIR[queries/]', '    QUERY_DIR --> OPENALEX[openalex.txt]', '    QUERY_DIR --> CROSSREF[crossref.json]', '    QUERY_DIR --> PUBMED[pubmed.txt]', '    QUERY_DIR --> EUROPEPMC[europepmc.txt]', '    QUERY_DIR --> SEMANTIC[semantic_scholar.txt]', '    PROJECT_ROOT --> DATABASE[project.db]', '    PROJECT_ROOT --> DATA[data/]', '    PROJECT_ROOT --> EXPORTS[exports/]', '    PROJECT_ROOT --> IMPORTS[imports/]', '    PROJECT_ROOT --> LOGS[logs/]', '    PROJECT_ROOT --> SCREENING[screening/]', '    PROJECT_ROOT --> OUTPUT[NandA-found-sources-on-clinical-reasoning-through-SLR.md]', '```', '', 'The configured notebook calls the three listed Python scripts. The other project folders preserve workflow artifacts for later import, screening, logging, data, and export stages; they are not evidence of completed screening in this report.', '']
appendix_lines = appendix_lines[:workflow_start] + data_science_appendix + appendix_lines[processing_start:]
lines.extend(appendix_lines)

report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(f'Exported {len(verified_records)} Crossref-verified APA references to {report_path}')

Exported 283 deduplicated candidate records to D:\OneDrive - Hogeschool Rotterdam\1_CURRENT_CODE\FLOWISE_DEEP_RESEARCH_GOOGLE\GOOGLE_API_KEY\GOOGLE_SCHOLAR\SLR-Engine\projects\NANDA\NandA-found-sources-on-clinical-reasoning-through-SLR.md
